# 21.15 — MP-03. Регистрация фаз 4D-КТ

**Статус: ограниченный прямой pilot MONAI Physio Greedy на двух парах одной КТ.** Исполнены `phase_00 → phase_01` и `phase_00 → phase_07` Георга, не все 15 фаз. Это кандидатные поля и внутренний QC, без независимой анатомической валидации и без передачи в FEM/УО/ФВ.

Связи: [план ветки](README.md), [парный код Greedy](mp03_04_monai_pair.py), [жёсткий контроль](mp03_04_pilot.py), [полный отчёт](mp03_04_report.md), [MP-04](21.16_MP04_Согласованная_4D_геометрия.ipynb), [официальный API Greedy](https://project-monai.github.io/monai-physio/api/registration/greedy.html).

## Основной расчёт на Нике: исходная сетка 4D-КТ, 28.09.2026

**Статус:** прямой технический расчёт на реальных фазах 00, 01 и 07 Ника, без независимой анатомической валидации. Предыдущие разделы о Георге сохранены как исторический пилот; основным индивидуальным результатом теперь является [отчёт Ника](mp03_nix_fullres_report.md). [Инвентаризация проектов Inobitec](mp03_nix_reference_inventory.md) подтверждает связь одного SPJ с 12 фазами DICOM, но не наличие принятой ручной разметки этих фаз.

КТ и автоматические `whole_heart` имеют исходную сетку `512×512×256`, шаг `0,765625×0,765625×0,625` мм. Crop `216×245×250` определён только по маске 00 с запасом 25 мм; CT-оптимизация не использовала целевую маску. Пары 00→01 и 00→07 обработаны Greedy без прореживания. В обоих полях итоговый sampled det J положителен после выбора сглаженного варианта 07; несглаженный 07 отклонён из-за 115 складок внутри auto-маски.

| Пара | Dice неподвижно → Greedy против auto-цели | ASSD, мм | Минимум det J | Решение |
|---|---:|---:|---:|---|
| 00→01 | 0,93002 → 0,95304 | 1,395 → 0,888 | 0,179 | Технический кандидат; повтор выявил небольшую межзапускную вариацию |
| 00→07, исходное сглаживание | 0,92654 → 0,96280 | 1,538 → 0,745 | −0,082 | Отклонён: 115 складок |
| 00→07, усиленное сглаживание | 0,92654 → 0,95905 | 1,538 → 0,826 | 0,547 | Технический кандидат |

На верхнем срезе всех трёх фаз наружная маска `whole_heart` касается границы FOV. Последующая [проверка трёх масок на всех фазах](mp03_nix_fov_contact_report.md) показала, что `blood_pool` и `myocardium` границы не касаются. По уточнению автора, 4D-КТ имеет меньший охват, чем статическая КТ; это касание само по себе не является проблемой. Маски TotalSegmentator v2 являются независимыми по фазе автоматическими выходами, но **не независимой ручной анатомической разметкой**. Геометрию нельзя допускать к FEM по одному улучшению Dice. Все плотные поля и изображения просмотра остаются локально в `runs/`.


## Дополнение 30.09.2026: все фазы Ника на исходной сетке

[Полный внутренний экран](mp03_nix_full_cycle_report.md) охватывает 11 пар `phase_00→phase_01…11` на исходном разрешении `512×512×256`, с одинаковым сглаживанием `3.0vox 1.0vox`. Dice к отдельным автоматическим целевым маскам вырос во всех парах в среднем на `0,0282`; минимум измеренного det J `0,410`, неположительных узлов в сохранённых полях нет. Почти нулевой выигрыш фазы 10 (`+0,0001` Dice) и остаточный Dice `0,901` фазы 03 записаны как ограничения. Старое поле 00→07 со 115 складками не включено.

Автоматические маски используются только для внутреннего QC. Независимой принятой ручной фазовой разметки нет. `whole_heart` касается верхнего края FOV во всех фазах из-за меньшего охвата 4D-КТ по сравнению со статической; [пофазная и вращаемая 3D-визуализация](mp03_nix_fov_contact_report.md) показывает это касание, которое само по себе не блокирует расчёт. Отдельные `blood_pool` и `myocardium` края не достигают. Без независимой анатомической проверки и оценки ошибки карты КТ→FEM поля не получили **анатомического допуска** и результаты не считаются измеренной точностью движения сердца. Позднее четыре поля использованы только в [исследовательском парном FEM-опыте](mp02_nix_paired_fem_report.md). [Пофазные обезличенные показатели](mp03_nix_full_cycle.json) сохраняют хэши и параметры.

## Входы, фазы и геометрия

**Факт о данных.** CT HU-NIfTI и `whole_heart` маски TotalSegmentator v2 — существующие продукты Georg. Маски не являются независимым референсом. Исходные данные остаются локально. Для каждой пары SHA-256 входов сохранены в [01 QC](mp03_04_georg_00_01_monai_qc.json) и [07 QC](mp03_04_georg_00_07_monai_qc.json). Номера фаз не означают ED/ES или подтверждённый общий R–R цикл; дыхательное состояние отдельно не подтверждено.

**Контроль единиц.** Заголовок NIfTI сообщает `unknown` для пространства. Миллиметры и RAS установлены по локальному `conversion_manifest.json` (`voxel_spacing_mm=[0.458984,0.458984,0.625]`, `affine_coordinate_system=NIfTI RAS+`) и сверены с affine. Все 15 CT имеют форму `512×512×256` и общий affine; выбранные маски им соответствуют. При подготовке ITK изображения переведены из RAS в LPS без изменения масштаба.

## Исполненный метод и направление

Версия `monai_physio=2026.09.1`. Официальный `RegisterImagesGreedy.register` использует `picsl_greedy` как in-memory нативный Python-модуль; отдельный `greedy.exe` не нужен. CT открыты nibabel и переданы ITK в памяти; обход Unicode-пути через `subst` не понадобился. CUDA была доступна, однако Greedy исполнялся на CPU с четырьмя потоками. Установка окружения не выполнялась.

Для каждой пары `fixed=цель`, `moving=phase_00`. Greedy: аффинная инициализация + деформируемый шаг, CC/WNCC, итерации `[12,6,3]`, сглаживание `2.0vox 0.5vox`. Маски определили общий crop 20 мм и использованы для оценки, **не** для подгонки. Шаг уменьшенной общей сетки около `1.377×1.377×1.25` мм. `fixed_to_moving` отображает физическую точку цели в источник (ITK LPS, мм) и нужен для pullback resampling маски/CT; `moving_to_fixed` переносит вершины источника в цель. В JSON сохранён контроль композиции этих преобразований.

## Сравнение на одной уменьшенной сетке

| Цель | Метод | Dice с отдельной auto mask | Ошибка центра, мм | Разница объёма, мл | ASSD / HD95, мм | CT-MSE, HU² |
|---|---|---:|---:|---:|---:|---:|
| 01 | Неподвижный | 0.96013 | 0.884 | +17.773 | 0.902 / 2.754 | 5394 |
| 01 | Жёсткий | 0.96013 | 0.884 | +17.773 | 0.902 / 2.754 | 4993 |
| 01 | Greedy | **0.97226** | **0.475** | **+2.725** | **0.637 / 1.860** | **1278** |
| 07 | Неподвижный | 0.94869 | **0.595** | +29.198 | 1.145 / 3.750 | 7017 |
| 07 | Жёсткий | 0.94869 | **0.595** | +29.198 | 1.145 / 3.750 | 6498 |
| 07 | Greedy | **0.96713** | 0.648 | **+2.266** | **0.722 / 2.314** | **1605** |

**Результат модели:** Greedy снизил CT-ошибку и расхождения масок на обеих парах. Для 07 ошибка центра стала **больше**. Эти числа измеряют согласие с отдельными автоматическими масками того же алгоритма, а не истинную ошибку движения. Старые полновоксельные значения жёсткого pilot не включены в эту таблицу из-за другого разрешения.

## Поле, складки и FOV

Плотные pullback-поля записаны только локально в исключённые из Git `runs/mp03_04_greedy_field_00_01.npz` и `runs/mp03_04_greedy_field_00_07.npz`. Оси массива `z,y,x,component_LPS_xyz`; вектор смещения задан в мм. Хэши полей и параметры сетки находятся в парных JSON.

| Цель | det J: минимум / p01 / медиана / p99 / максимум | Доля det J ≤ 0 | P95 смещения, мм | Pullback за moving crop | Ошибка fwd→inv: медиана / макс, мм |
|---|---|---:|---:|---:|---:|
| 01 | 0.419 / 0.769 / 1.000 / 1.301 / 2.168 | 0 | 1.951 | 0.223% | 0.016 / 0.069 |
| 07 | 0.347 / 0.702 / 1.000 / 1.357 / 1.983 | 0 | 3.327 | 0.354% | 0.025 / 0.089 |

**Граница вывода:** отсутствие складок подтверждено только в отсчётах уменьшенной сетки. Исходные маски лежат внутри полного FOV; crop доходит до верхней границы CT. Доли выхода относятся к crop. Первая пара при повторении с теми же параметрами дала близкие, но не одинаковые Dice (0.97219 и 0.97226) и минимум Якобиана (0.445 и 0.419); побитовая воспроизводимость backend не установлена.

In [ ]:
# Проверка сохранённых агрегатов без чтения CT и масок.
import json
from pathlib import Path

for target in ("01", "07"):
    d = json.loads(Path(f"mp03_04_georg_00_{target}_monai_qc.json").read_text(encoding="utf-8"))
    assert d["software"]["monai_physio"] == "2026.09.1"
    assert d["fixed_target"] == f"phase_{target}"
    assert d["field_qc"]["jacobian_of_pullback"]["nonpositive_fraction"] == 0
    q = d["mask_qc"]
    print(target, "Dice", round(q["stationary"]["dice_vs_auto_mask"], 5),
          "->", round(q["greedy"]["dice_vs_auto_mask"], 5),
          "center_mm", round(q["greedy"]["center_difference_mm"], 3),
          "min_jac", round(d["field_qc"]["jacobian_of_pullback"]["minimum"], 3))

## Воспроизведение и решение

Запускать **по одной паре** из папки `monai_physio/`: `<VENV_PYTHON> -B mp03_04_monai_pair.py --phase-dir <DERIVED>/georg/phases --products-dir <DERIVED>/georg/products_v2 --conversion-manifest <DERIVED>/georg/phases/conversion_manifest.json --reference-phase phase_00 --target-phase phase_01 --rigid-qc mp03_04_georg_00_01_qc.json --output-json mp03_04_georg_00_01_monai_qc.json --field-output runs/mp03_04_greedy_field_00_01.npz`. Для пары 07 заменить идентификатор в target и именах двух выходов/rigid-QC. Локальный `<DERIVED>` не встраивать в ноутбук. Этот код сохраняет поле только в исключённой из Git папке.

**Нерешённые вопросы:** independent reference отсутствует, на исходном разрешении две пары проверены в следующем разделе; остаются независимая анатомия, границы, повторяемость, фазовое время и влияние на FEM. Две пары не доказывают пригодность для остальных фаз. Статус поля — кандидат, не `accepted`.

## Исходное разрешение: отдельный проверенный прогон 28.09.2026

**Факт расчёта.** На реальной 4D-КТ Георга заново исполнены `00→01` и `00→07` с исходным шагом `0,458984×0,458984×0,625` мм на crop `414×441×235` без прореживания. Crop определён только маской 00 с запасом 25 мм; автоматическая маска цели использована для внутренней оценки. Параметры, команды, локальные выходы и ограничения: [полный отчёт](mp03_fullres_report.md). Исходные медицинские массивы и поля остаются вне Git.

| Поле | Dice auto-цели, неподвижно → зарегистрировано | ASSD, мм | CT-MSE, HU² | Минимум det J / число неположительных узлов |
|---|---:|---:|---:|---:|
| 00→01, стандартное сглаживание | 0,95965 → 0,97043 | 0,732 → 0,502 | 4504 → 938 | 0,276 / 0 |
| 00→07, стандартное сглаживание | 0,94898 → 0,96074 | 0,951 → 0,676 | 5088 → 1348 | −0,036 / 3 |
| 00→07, усиленное сглаживание | 0,94898 → 0,95950 | 0,951 → 0,702 | 5088 → 1596 | 0,466 / 0 |

Исходное поле 07 со складками отклонено. Более сглаженное поле устранило складки на узлах исходной сетки ценой небольшого ухудшения Dice/CT-MSE относительно первого варианта. У обоих технически положительных полей **нет независимого анатомического допуска**. [Инвентаризация](mp03_manual_reference_inventory.md) не установила ручной референс для этих фаз, а слабый контраст ограничивает его получение. Улучшение относительно другой автоматической маски не доказывает точность перемещения сердца или пригодность для FEM.


In [ ]:
# Локальная сверка агрегатов; нужна копия исключённого из Git runs/.
import json
from pathlib import Path

for suffix in ("00_01", "00_07", "00_07_smooth"):
    d = json.loads(Path(f"runs/mp03_fullres_{suffix}_qc.json").read_text(encoding="utf-8"))
    assert d["grid_qc"]["stride_ijk"] == [1, 1, 1]
    assert d["grid_qc"]["crop_source_only"]
    print(suffix, d["mask_qc"]["greedy"]["dice_vs_auto_mask"],
          d["field_qc"]["jacobian_of_pullback"]["minimum"])
